[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-00-record-and-replay-agent-tests.ipynb)

# Advanced Discussion 0 — testing an agent without calling a model

A record/replay harness for tool calls, a stand-in model, and what replay does and does not catch.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

No network or API key needed; runs in seconds.

In [ ]:
import json, time, hashlib, random, os, tempfile

## 1. a tiny agent with a stand-in "model": the point is the harness, so the model is a deterministic function of the conversation

In [ ]:
TOOLS_LIVE_CALLS = {"n": 0}
def live_search_orders(customer):
    TOOLS_LIVE_CALLS["n"] += 1; time.sleep(0.02)                      # a real tool: network latency
    return {"alice": ["ORD-1001", "ORD-1002"], "bob": ["ORD-2001"]}.get(customer, [])
def live_get_order(order_id):
    TOOLS_LIVE_CALLS["n"] += 1; time.sleep(0.02)
    return {"ORD-1001": {"status": "delivered", "total": 40.0}, "ORD-1002": {"status": "delayed", "total": 120.0}, "ORD-2001": {"status": "shipped", "total": 15.0}}.get(order_id, {"error": "not found"})
LIVE = {"search_orders": live_search_orders, "get_order": live_get_order}

def fake_model(messages, schema_version="v1"):
    """Stand-in LLM: reads the last message and decides the next action. Deterministic on purpose."""
    last = messages[-1]
    if last["role"] == "user":
        cust = last["content"].split("for ")[-1].strip("? ."); return {"tool": "search_orders", "args": {"customer": cust}}
    if last["role"] == "tool" and last["name"] == "search_orders":
        ids = json.loads(last["content"]); return {"tool": "get_order", "args": {"order_id": ids[-1]}} if ids else {"final": "No orders found."}
    if last["role"] == "tool" and last["name"] == "get_order":
        o = json.loads(last["content"]); return {"final": "Your latest order is %s." % o.get("status", "unknown")}
    return {"final": "I am not sure."}

class Recorder:
    """Wrap the tools: in 'record' mode call the live tool and store (tool, args) -> result; in 'replay' mode never touch the network."""
    def __init__(self, path, mode): self.path, self.mode = path, mode; self.tape = json.load(open(path)) if mode == "replay" and os.path.exists(path) else {}; self.hits = 0
    def key(self, name, args): return name + ":" + json.dumps(args, sort_keys=True)
    def call(self, name, args):
        k = self.key(name, args)
        if self.mode == "replay":
            if k not in self.tape: raise KeyError("no recording for %s - the agent asked for a call the tape does not contain" % k)
            self.hits += 1; return self.tape[k]
        r = LIVE[name](**args); self.tape[k] = r; return r
    def save(self): json.dump(self.tape, open(self.path, "w"), indent=1)

def run_agent(question, rec, model=fake_model, max_steps=6):
    messages = [{"role": "user", "content": question}]; trace = []
    for _ in range(max_steps):
        act = model(messages)
        if "final" in act: return act["final"], trace
        res = rec.call(act["tool"], act["args"]); trace.append((act["tool"], act["args"]))
        messages.append({"role": "tool", "name": act["tool"], "content": json.dumps(res)})
    return "gave up", trace

cases = [("What is the status of the latest order for alice?", "Your latest order is delayed."), ("What is the status of the latest order for bob?", "Your latest order is shipped."),
         ("What is the status of the latest order for carol?", "No orders found.")]
tape = os.path.join(tempfile.mkdtemp(), "tape.json")
# record once (live tools), then replay many times
t0 = time.perf_counter(); rec = Recorder(tape, "record")
for q, want in cases: ans, tr = run_agent(q, rec); print("record : %-55s -> %-32s tools: %s" % (q, ans, [t[0] for t in tr]))
rec.save(); t_rec = time.perf_counter() - t0; live_calls = TOOLS_LIVE_CALLS["n"]
print("recording used %d live tool calls in %.3f s; tape holds %d entries" % (live_calls, t_rec, len(rec.tape)))
TOOLS_LIVE_CALLS["n"] = 0; t0 = time.perf_counter(); passed = 0
for _ in range(100):
    rp = Recorder(tape, "replay")
    for q, want in cases: ans, tr = run_agent(q, rp); passed += ans == want
t_rep = time.perf_counter() - t0
print("replay : 100 x %d test cases = %d checks, %d passed, %d live tool calls, %.3f s total (%.3f ms per case)" % (len(cases), 100 * len(cases), passed, TOOLS_LIVE_CALLS["n"], t_rep, 1000 * t_rep / (100 * len(cases))))
print("the same 300 cases against the live tools would have taken about %.1f s and made %d real calls" % (t_rec / len(cases) * 300, 300 * live_calls // len(cases)))

## 2. what replay catches: change the agent and re-run the tape

In [ ]:
def model_v2_skips_lookup(messages):        # a 'prompt change' that makes the agent guess instead of calling get_order
    last = messages[-1]
    if last["role"] == "tool" and last["name"] == "search_orders": return {"final": "Your latest order is on its way."}
    return fake_model(messages)
def model_v3_new_arg_name(messages):        # a refactor that renames a tool argument
    act = fake_model(messages)
    if act.get("tool") == "get_order": act["args"] = {"id": act["args"]["order_id"]}
    return act
def model_v4_extra_call(messages):          # an extra lookup before answering
    last = messages[-1]
    if last["role"] == "tool" and last["name"] == "get_order" and not any(m.get("name") == "search_orders" and m["role"] == "tool" and m.get("dup") for m in messages):
        pass
    return fake_model(messages)
print("\nchange the agent, replay the same tape:")
for name, mdl in (("unchanged agent", fake_model), ("v2: answers without looking the order up", model_v2_skips_lookup), ("v3: renamed a tool argument", model_v3_new_arg_name)):
    outcomes = []
    for q, want in cases:
        try: ans, tr = run_agent(q, Recorder(tape, "replay"), model=mdl); outcomes.append("ok" if ans == want else "WRONG ANSWER (%s)" % ans)
        except KeyError as e: outcomes.append("TAPE MISS")
    print("  %-44s %s" % (name, " | ".join(outcomes)))
print("a wrong answer is caught by the assertion; a call pattern the tape does not contain is caught as a tape miss. Neither needed a model API or a live tool.")

## 3. what replay cannot catch

In [ ]:
print("\nwhat replay does NOT test: whether the real model still chooses these calls (it tests the harness, tool code and your assertions, not the live model), whether the live tool's data or schema has changed since the recording, and anything time-dependent. Refresh tapes on a schedule and keep a small live smoke test.")

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.